In [1]:
import pandas as pd

df = pd.read_csv("../data/delhi_ncr_aqi_dataset.csv")
print(df.shape)
df.head()

(201664, 25)


,datetime,date,year,month,day,hour,day_of_week,is_weekend,season,city,...,no2,so2,co,o3,temperature,humidity,wind_speed,visibility,aqi,aqi_category
0,2020-01-01 06:00:00,2020-01-01,2020,1,1,6,Wednesday,0,winter,Delhi,...,119.6,47.7,5.19,12.3,9.4,100,3.6,1.2,500,Severe
1,2020-01-01 12:00:00,2020-01-01,2020,1,1,12,Wednesday,0,winter,Delhi,...,117.9,39.3,4.32,15.8,20.6,50,5.9,1.4,500,Severe
2,2020-01-01 18:00:00,2020-01-01,2020,1,1,18,Wednesday,0,winter,Delhi,...,150.1,36.3,7.13,14.3,12.4,56,4.5,1.1,500,Severe
3,2020-01-01 23:00:00,2020-01-01,2020,1,1,23,Wednesday,0,winter,Delhi,...,142.0,30.3,4.90,13.2,14.4,48,5.8,1.4,500,Severe
4,2020-01-01 06:00:00,2020-01-01,2020,1,1,6,Wednesday,0,winter,Delhi,...,138.4,41.5,7.56,15.4,6.8,100,2.8,0.4,500,Severe


In [2]:
df = pd.read_csv("../data/delhi_ncr_aqi_dataset.csv")

cols_to_drop = ['datetime', 'date', 'station', 'latitude', 'longitude']
df = df.drop(columns=cols_to_drop)

print("Columns remaining:", df.shape[1])
print(list(df.columns))

Columns remaining: 20
['year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend', 'season', 'city', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'temperature', 'humidity', 'wind_speed', 'visibility', 'aqi', 'aqi_category']


In [3]:
day_map = {
    'Monday': 0,
    'Tuesday': 1,
    'Wednesday': 2,
    'Thursday': 3,
    'Friday': 4,
    'Saturday': 5,
    'Sunday': 6
}
df['day_of_week'] = df['day_of_week'].map(day_map)
print(df['day_of_week'].unique())

[2 3 4 5 6 0 1]


In [4]:
season_map = {
    'monsoon': 0,
    'summer': 1,
    'post_monsoon': 2,
    'winter': 3
}
df['season'] = df['season'].map(season_map)
print(df['season'].unique())


[3 1 0 2]


In [5]:
from sklearn.preprocessing import LabelEncoder
le_city = LabelEncoder()
df['city'] = le_city.fit_transform(df['city'])
print(df['city'].unique())
print(le_city.classes_)

[0 4 3 1 2]
['Delhi' 'Faridabad' 'Ghaziabad' 'Gurugram' 'Noida']


In [6]:
df[['day_of_week', 'season', 'city']].head(8)

,day_of_week,season,city
0,2,3,0
1,2,3,0
2,2,3,0
3,2,3,0
4,2,3,0
5,2,3,0
6,2,3,0
7,2,3,0


In [7]:
cat_map = {
    'Good': 0,
    'Satisfactory': 1,
    'Moderate': 2,
    'Poor': 3,
    'Very Poor': 4,
    'Severe': 5
}
df['aqi_category_encoded'] = df['aqi_category'].map(cat_map)
print(df[['aqi_category', 'aqi_category_encoded']].head(8))

  aqi_category  aqi_category_encoded
0       Severe                     5
1       Severe                     5
2       Severe                     5
3       Severe                     5
4       Severe                     5
5    Very Poor                     4
6       Severe                     5
7       Severe                     5


In [8]:
print(df.dtypes)

year                      int64
month                     int64
day                       int64
hour                      int64
day_of_week               int64
is_weekend                int64
season                    int64
city                      int64
pm25                    float64
pm10                    float64
no2                     float64
so2                     float64
co                      float64
o3                      float64
temperature             float64
humidity                  int64
wind_speed              float64
visibility              float64
aqi                       int64
aqi_category             object
aqi_category_encoded      int64
dtype: object


In [9]:
X = df.drop(columns=['aqi', 'aqi_category', 'aqi_category_encoded'])
y_reg = df['aqi']
y_clf = df['aqi_category_encoded']
print("X shape:" , X.shape)
print("y_reg shape:" , y_reg.shape)
print("y_clf shape:" , y_clf.shape)

X shape: (201664, 18)
y_reg shape: (201664,)
y_clf shape: (201664,)


In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)
print("Training rows: ", X_train.shape[0])
print("Test rows: ", X_test.shape[0])

Training rows:  161331
Test rows:  40333


In [11]:
X_train, X_test, y_train_clf, y_test_clf = train_test_split(
    X, y_clf,
    test_size=0.2,
    random_state=42
)

print("y_train_clf shape:", y_train_clf.shape)
print("y_test_clf shape:", y_test_clf.shape)

y_train_clf shape: (161331,)
y_test_clf shape: (40333,)


In [12]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Done! First row of X_train_scaled:")
print(X_train_scaled[0])

Done! First row of X_train_scaled:
[ 0.29491532  0.71930199 -0.65085613 -0.43003619  1.49997982  1.58016863
 -1.12334701 -0.70142382 -0.6225298  -0.62797018 -0.67367696 -0.66291376
 -0.66970083  0.20047544  1.10225842  0.72327286  0.30955288  0.04956943]


In [13]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')
print("Scaler saved!")

Scaler saved!
